# 03 · Urban data management

*Notebooks: [01 Overview](01-overview.ipynb) · [02 Formats](02-formats.ipynb) · [03 Data](03-data.ipynb) · [04 Maps](04-maps.ipynb) · [05 Plots](05-plots.ipynb) · [06 Compute](06-compute.ipynb)*

In the JavaScript tutorial this part is `autk-db`: a DuckDB database in the browser, with workspaces, loaders and SQL.
pyautark exposes the same engine declaratively. You list the **sources**, then the **transforms** that derive tables from
them, and the engine loads and runs them in order, in the browser. This notebook loads each source type, joins tables
spatially and builds a heatmap. It ends with what to do in Python where the JavaScript tutorial uses SQL.

In [ ]:
import autark as ak
import tutorial as t

## 1 Tables and workspaces

Every source becomes a table named by its `name`. Layers, plots and transforms refer to tables by that name. Autark keeps
all tables in one workspace and holds geometry in World Mercator meters (EPSG:3395), so distances you pass (`Near`) are
in meters. Sources are loaded in the order of `data`, and **the first one sets the extent of the workspace**: later
sources are cut to it, so a table that lies outside the first one comes out empty. Put the source that covers the area
of interest first.

## 2 Loading data

### 2.1 OpenStreetMap

An `OSM` source with `layers=[...]` creates one table per layer (`<name>_<layer>`): `surface`, `parks`, `water`,
`roads` and `buildings`. The building table has the heights and levels OSM carries.

In [ ]:
osm = ak.OSM(
    "mnt", bbox=(-74.02, 40.700, -73.995, 40.72), layers=["roads", "buildings"],
    source="pbf", pbf_file_url=t.url("data/osm/lower_mnt.osm.pbf"),
)

ak.Spec(
    data=[osm],
    views=[
        ak.Map(layers=[ak.Layer("mnt_roads", id="roads").style(color="#888888"), ak.Layer("mnt_buildings", id="buildings")]),
        ak.Table("mnt_buildings", columns=["name", "building", "height"]),
    ],
).widget()

### 2.2 GeoJSON, 2.3 CSV with WKT, 2.4 GeoTIFF

The same pattern for the other formats. The table below is a plot of the neighborhoods' properties: a `Table` view reads
any table by name.

In [ ]:
neighborhoods = ak.GeoJSON("neighborhoods", url=t.url("data/map/mnt_neighs.geojson"))
island = ak.CSV("island", url=t.url("data/db/mnt.csv"), geometry=ak.wkt("wkt"))
noise = ak.CSV("noise", url=t.url("data/map/mnt_noise.csv"), geometry=ak.latlng("latitude", "longitude"))

ak.Spec(
    data=[neighborhoods, island, noise],
    views=[
        ak.Map(layers=[ak.Layer(island).style(color="#cccccc"), ak.Layer(neighborhoods, id="neighborhoods").style(opacity=0.6)]),
        ak.Table(neighborhoods, columns=["ntaname", "boroname", "shape_area"], sort={"column": "shape_area", "direction": "desc"}),
    ],
    layout=ak.Layout(type="vertical"),
).widget()

## 3 Filtering and grouping

`autk-db` answers SQL. pyautark has no SQL in the spec, by design: a spec is data that any host can run, and filtering
and grouping are one line in pandas. Do them before the data goes in, or let a spatial join aggregate (below).

For example, noise complaints at night, as a new inline source:

In [ ]:
import pandas as pd

complaints = pd.read_csv(t.ROOT / "data/map/mnt_noise.csv", parse_dates=["date"])
at_night = complaints[(complaints["date"].dt.hour >= 22) | (complaints["date"].dt.hour < 5)]
print(len(complaints), "complaints,", len(at_night), "at night")
print(at_night.groupby(at_night["date"].dt.hour).size())

In [ ]:
night = ak.GeoJSON.from_dataframe("night", at_night, latitude="latitude", longitude="longitude", properties=["key"])

ak.Spec(
    data=[neighborhoods, night],
    views=[ak.Map(layers=[ak.Layer(neighborhoods).style(color="#dddddd"), ak.Layer(night).style(color="#e45756", opacity=0.6)])],
).widget()

## 4 Spatial queries

### 4.1 Spatial join

`ak.SpatialJoin(root=..., join=..., group_by=[...])` adds to each feature of `root` an aggregation of the features of
`join` that fall inside it (or within `near`). The result **replaces the root table**: its features gain properties
named `sjoin.<function>.<table>`, here `sjoin.count.noise`. A later layer or plot reads it with that path.

In [ ]:
join = ak.SpatialJoin(root=neighborhoods, join=noise, group_by=[ak.count()])

spec = ak.Spec(
    data=[neighborhoods, noise],
    transforms=[join],
    views=[
        ak.Map(layers=[
            ak.Layer(neighborhoods, id="neighborhoods").encode(
                color=ak.field("sjoin.count.noise", ak.Scale(scheme="reds", type="quantile"))
            ),
        ]),
        ak.Table(neighborhoods, columns=["ntaname", "sjoin.count.noise"], sort={"column": "sjoin.count.noise", "direction": "desc"}),
    ],
    layout=ak.Layout(type="vertical"),
)
spec.widget()

Other aggregations are `ak.total`, `ak.avg`, `ak.minimum`, `ak.maximum`, `ak.weighted` and `ak.collect`, each with
`normalize=True` to scale the result to 0 to 1. `near=ak.Near(distance)` joins the features within `distance` meters
instead of the ones inside. Here is the spec's JSON for the join:

In [ ]:
join.to_dict()

### 4.2 Heatmaps

`ak.Heatmap` aggregates a table onto a regular grid, producing a **raster** table. The grid has `rows` by `columns`
cells; each cell aggregates the points within `near` meters of its center. The raster is a new table, named by
`output`, and a layer colors it by the value `count.noise`.

In [ ]:
heat = ak.Heatmap(
    noise, output="noise_heat", near=ak.Near(800),
    grid=ak.HeatmapGrid(rows=40, columns=40), group_by=[ak.count()],
)

ak.Spec(
    data=[neighborhoods, noise],
    transforms=[heat],
    views=[ak.Map(layers=[
        ak.Layer(neighborhoods, id="neighborhoods").style(color="#eeeeee"),
        ak.Layer("noise_heat", id="heatmap").encode(color=ak.field("count.noise", ak.Scale(scheme="inferno"))).style(opacity=0.7),
    ])],
).widget()

Two rules the engine enforces: sources and joins and heatmaps run **before** compute passes, so a join cannot read a table a
compute pass wrote (pyautark raises an error); and a heatmap cannot `collect`.